# ***元类***

***1. type()***
> type()函数既可以查看对象类型，又可以创建出新的对象, 而无需通过class Hello(object)...的定义
>
> 要创建一个class对象，type()函数依次传入3个参数
> > 1) class的名称
> >
> > 2) 继承的父类集合(tuple格式，注意单元素tuple的写法)
> >
> > 3) class的方法名称与函数绑定
>
> Python解释器遇到class定义时，扫描class定义的语法，然后调用type()函数创建出class
> >


In [1]:
def fn(self, name='world'): # 先定义函数
    print(f'Hello, {name}.')
def fn2(self, name='world'):
    print(f'Goodbye, {name}')
Hello = type('Hello', (object,), dict(hello=fn, goodbye=fn2)) # 创建Hello class
h = Hello()
h.hello()
h.goodbye()

print(type(Hello))

print(type(h))



Hello, world.
Goodbye, world
<class 'type'>
<class '__main__.Hello'>


***2. metaclass***
> 根据类创建实例，根据 `metaclass` 创建类
>
> `metaclass` 是类的模板，所以必须从 `type` 类型派生
>
> 传入关键字参数 `metaclass` 时，魔术就生效了，它指示Python解释器在创建 `MyList` 时，要通过 `ListMetaclass.__new__()` 来创建
>

In [6]:
# metaclass是类的模板，所以必须从`type`类型派生：
class ListMetaclass(type):
    def __new__(cls, name, bases, attrs):
        attrs['add'] = lambda self, value: self.append(value)
        return type.__new__(cls, name, bases, attrs)
class MyList(list, metaclass=ListMetaclass):
    pass

L = MyList()
L.add(1) # list本身是没有add方法的，只有append
print(L, type(L), issubclass(MyList, list))

[1] <class '__main__.MyList'> True


***3. ORM***

- `Field`类，它负责保存数据库表的字段名和字段类型

In [3]:
class Field(object):

    def __init__(self, name, column_type):
        self.name = name
        self.column_type = column_type

    def __str__(self):
        return '<%s:%s>' % (self.__class__.__name__, self.name)


- 在 `Field` 的基础上，进一步定义各种类型的 `Field`
- `super(StringField, self)` 会根据类的继承顺序（MRO，方法解析顺序）找到 `StringField` 的直接父类 `Field`

In [4]:
class StringField(Field):
    def __init__(self, name):
        # 等效于 Field.__init__(self, name, 'varchar(100)')
        # super(StringField, self).__init__(name, 'varchar(100)')
        super().__init__(name, 'varchar(100)')


class IntegerField(Field):
    def __init__(self, name):
        super(IntegerField, self).__init__(name, 'bigint')


- `ModelMetaclass`

In [6]:
class ModelMetaclass(type):
    def __new__(cls, name, bases, attrs):
        if name=='Model':
            return type.__new__(cls, name, bases, attrs)
        print('Found model: %s' % name)
        mappings = dict()
        for k, v in attrs.items():
            if isinstance(v, Field):
                print('Found mapping: %s ==> %s' % (k, v))
                mappings[k] = v
        for k in mappings.keys():
            attrs.pop(k)
        attrs['__mappings__'] = mappings # 保存属性和列的映射关系
        attrs['__table__'] = name # 假设表名和类名一致
        return type.__new__(cls, name, bases, attrs)


- 基类

In [8]:
class Model(dict, metaclass=ModelMetaclass):
    def __init__(self, **kw):
        super(Model, self).__init__(**kw)

    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(r"'Model' object has no attribute '%s'" % key)

    def __setattr__(self, key, value):
        self[key] = value

    def save(self):
        fields = []
        params = []
        args = []
        for k, v in self.__mappings__.items():
            fields.append(v.name)
            params.append('?')
            args.append(getattr(self, k, None))
        sql = 'insert into %s (%s) values (%s)' % (self.__table__, ','.join(fields), ','.join(params))
        print('SQL: %s' % sql)
        print('ARGS: %s' % str(args))


In [9]:
class User(Model):
    # 定义类的属性到列的映射：
    id = IntegerField('id')
    name = StringField('username')
    email = StringField('email')
    password = StringField('password')

# 创建一个实例：
u = User(id=12345, name='Michael', email='test@orm.org', password='my-pwd')
# 保存到数据库：
u.save()


Found model: User
Found mapping: id ==> <IntegerField:id>
Found mapping: name ==> <StringField:username>
Found mapping: email ==> <StringField:email>
Found mapping: password ==> <StringField:password>
SQL: insert into User (id,username,email,password) values (?,?,?,?)
ARGS: [12345, 'Michael', 'test@orm.org', 'my-pwd']
